In [1]:
import pandas as pd
from pathlib import Path

In [2]:
ruta = r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\01_raw\indicadores sociales\desigualdad\60143.csv"

df = pd.read_csv(
    ruta,
    encoding='latin-1',
    sep=None,
    engine='python'
)

print(df.shape)
df.head()

(1440, 4)


,Comunidades y Ciudades Autónomas,Desigualdad en la distribución de ingresos (S80/S20 y coeficiente de Gini),Periodo,Total
0,Total Nacional,Desigualdad (S80/S20),2025,"5,2"
1,Total Nacional,Desigualdad (S80/S20),2024,"5,4"
2,Total Nacional,Desigualdad (S80/S20),2023,"5,5"
3,Total Nacional,Desigualdad (S80/S20),2022,"5,6"
4,Total Nacional,Desigualdad (S80/S20),2021,"6,2"


Comprobamos que todo está bien escrito para hacer la normalización correspondiente. 

In [3]:
print(sorted(df['Comunidades y Ciudades Autónomas'].unique()))
print()
print(df['Desigualdad en la distribución de ingresos (S80/S20 y coeficiente de Gini)'].unique())

['01 Andalucía', '02 Aragón', '03 Asturias, Principado de', '04 Balears, Illes', '05 Canarias', '06 Cantabria', '07 Castilla y León', '08 Castilla - La Mancha', '09 Cataluña', '10 Comunitat Valenciana', '11 Extremadura', '12 Galicia', '13 Madrid, Comunidad de', '14 Murcia, Región de', '15 Navarra, Comunidad Foral de', '16 País Vasco', '17 Rioja, La', '18 Ceuta', '19 Melilla', 'Total Nacional']

['Desigualdad (S80/S20)'
 'Distribución de la renta S80/S20 (alquiler imputado)' 'Gini'
 'Gini (con alquiler imputado)']


Nos quedamos con los datos más limpios y comparables, y eliminamos los agregados que no encajarán cuando unamos tablas.

In [4]:
df = df[
    (df['Comunidades y Ciudades Autónomas'] != 'Total Nacional') &
    (df['Desigualdad en la distribución de ingresos (S80/S20 y coeficiente de Gini)']
     .isin(['Desigualdad (S80/S20)', 'Gini']))
].copy()

print(df.shape)

(684, 4)


Cambiamos el nombre de la columna para que todas las columnas tengan el mismo nombre y así poder hacer un merge posteriormente.

In [5]:
# Renombrar columnas
df = df.rename(columns={
    'Comunidades y Ciudades Autónomas': 'comunidad',
    'Desigualdad en la distribución de ingresos (S80/S20 y coeficiente de Gini)': 'indicador',
    'Periodo': 'año',
    'Total': 'valor'
})

print(df.columns.tolist())

['comunidad', 'indicador', 'año', 'valor']


Limpiamos el nombre de las comunidades, quitamos el prefijo y lo cambiamos por el mismo nombre que en la tabla de delitos de odio

In [6]:
# Limpiar prefijo numérico y nombres largos
df['comunidad'] = (
    df['comunidad']
    .str.replace(r'^\d+\s+', '', regex=True)
    .str.strip()
    .str.replace('Asturias, Principado de', 'Asturias', regex=False)
    .str.replace('Balears, Illes', 'Baleares', regex=False)
    .str.replace('Castilla - La Mancha', 'Castilla-La Mancha', regex=False)
    .str.replace('Comunitat Valenciana', 'Comunidad Valenciana', regex=False)
    .str.replace('Madrid, Comunidad de', 'Madrid', regex=False)
    .str.replace('Murcia, Región de', 'Murcia', regex=False)
    .str.replace('Navarra, Comunidad Foral de', 'Navarra', regex=False)
    .str.replace('Rioja, La', 'La Rioja', regex=False)
)

print(sorted(df['comunidad'].unique()))

['Andalucía', 'Aragón', 'Asturias', 'Baleares', 'Canarias', 'Cantabria', 'Castilla y León', 'Castilla-La Mancha', 'Cataluña', 'Ceuta', 'Comunidad Valenciana', 'Extremadura', 'Galicia', 'La Rioja', 'Madrid', 'Melilla', 'Murcia', 'Navarra', 'País Vasco']


Cambiamos el tipo de la columna 'valor'

In [7]:
df['valor'] = (
    df['valor'].astype(str)
    .str.replace(',', '.', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
)

print(df.dtypes)
print(df['valor'].isna().sum(), "nulos")
df.head(10)

comunidad     object
indicador     object
año            int64
valor        float64
dtype: object
0 nulos


,comunidad,indicador,año,valor
72,Andalucía,Desigualdad (S80/S20),2025,5.4
73,Andalucía,Desigualdad (S80/S20),2024,5.5
74,Andalucía,Desigualdad (S80/S20),2023,5.9
75,Andalucía,Desigualdad (S80/S20),2022,6.0
76,Andalucía,Desigualdad (S80/S20),2021,6.8
77,Andalucía,Desigualdad (S80/S20),2020,5.3
78,Andalucía,Desigualdad (S80/S20),2019,6.1
79,Andalucía,Desigualdad (S80/S20),2018,6.5
80,Andalucía,Desigualdad (S80/S20),2017,6.9
81,Andalucía,Desigualdad (S80/S20),2016,7.2


Guardamos la tabla ya limpia

In [8]:
import os

RUTA_PROCESADOS = r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13\01_data\02_processed\indicadores_sociales\desigualdad"
os.makedirs(RUTA_PROCESADOS, exist_ok=True)

df.to_csv(
    RUTA_PROCESADOS + r"\desigualdad_españa.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"Guardado correctamente: {len(df):,} filas")

Guardado correctamente: 684 filas
